# Method of Lines and Runge-Kutta

Connect spatial right-hand sides to Runge-Kutta time stages using NRPy time-stepping utilities.

Navigation: [Index](../index.ipynb) | Previous: [Wave Equation with NumPy](wave_equation_with_numpy.ipynb) | Next: [Boundary Conditions and Convergence](boundary_conditions_and_convergence.ipynb)


## RK4 Stages and NRPy Registration
The right-hand side computes time derivatives. RK4 combines four stage evaluations into one time step.

## Import MoL Registry Tools

These imports expose the NRPy registries and infrastructure writers used below.


In [ ]:
import nrpy.c_function as cfc
from nrpy.infrastructures import BHaH
from nrpy.infrastructures.BHaH.MoLtimestepping.MoL_step_forward_in_time import (
    register_CFunction_MoL_step_forward_in_time,
)
from nrpy.infrastructures.BHaH.MoLtimestepping.rk_butcher_table_dictionary import (
    generate_Butcher_tables,
    intermediate_stage_gf_names_list,
    is_diagonal_Butcher,
    validate,
)


## Inspect the RK4 Butcher Table

The output shows the Runge-Kutta or Method of Lines objects used by generated code.


In [ ]:
butcher_tables = generate_Butcher_tables()
validate(butcher_tables, ["RK4"], verbose=False)
rk4_table, rk4_order = butcher_tables["RK4"]
print("RK4 order:", rk4_order)
print("is diagonal:", is_diagonal_Butcher(butcher_tables, "RK4"))
print(
    "intermediate storage names:",
    intermediate_stage_gf_names_list(butcher_tables, "RK4"),
)


## Register the RK Step-Forward Function

The registry stores a complete C function or infrastructure hook for later file generation.


In [ ]:
cfc.CFunction_dict.pop("MoL_step_forward_in_time", None)
for name in ["RK_STEP1", "RK_STEP2", "RK_STEP3", "RK_STEP4"]:
    BHaH.MoLtimestepping.rk_substep.MoL_Functions_dict.pop(name, None)
register_CFunction_MoL_step_forward_in_time(
    butcher_tables,
    "RK4",
    rhs_string="rhs_eval(commondata, params, rfmstruct, auxevol_gfs, in_gfs, rhs_gfs);",
)
registered = cfc.CFunction_dict["MoL_step_forward_in_time"]
print(
    "registered MoL function names:",
    sorted(name for name in cfc.CFunction_dict if "MoL" in name or "mol" in name),
)
print("complete prototype:")
print(registered.function_prototype)
print(
    "registered substep names:",
    sorted(BHaH.MoLtimestepping.rk_substep.MoL_Functions_dict),
)


## Inspect the Generated RK Substep

The complete block shows how the Method of Lines update is written as a C helper function.


In [ ]:
full_function = registered.full_function
start = full_function.index("static void rk_substep_1_host")
end_marker = "} // END FUNCTION: rk_substep_1_host"
end = full_function.index(end_marker, start) + len(end_marker)
print("complete generated RK substep block:")
print(full_function[start:end])


The RK4 validation confirms the stage table, and the generated substep block shows how NRPy turns those stages into C. This is the time-integration driver used by generated wave projects.


## Continue to Boundary Conditions
- [Boundary Conditions and Convergence](boundary_conditions_and_convergence.ipynb)
- [Start-to-Finish Cartesian Wave Project](../3-wave_equation/start_to_finish_wave_cartesian.ipynb)
- [BHaH Project Anatomy](../5-infrastructures/bhah_project_anatomy.ipynb)
